# Stage 5 - adversarial training under two threat models (decisive, averaged)

Both datasets are evaluated the SAME way, under BOTH threat models, and averaged over repeats.

- **transfer**: attack crafted from the *baseline* CNN, fed to every model (a real attacker has no access to the defended model).
- **white-box**: attack crafted against the model it hits (worst case; the Random Forest has no gradients, so white-box does not apply to it).

The metric is robust-support macro-F1 (classes with >=2 test frames, computed the same way for both datasets). CICIoV single runs are unstable on tiny data, so the averaged table below is the one to read; the first cell just shows one run in detail.

In [1]:
import sys
from pathlib import Path
try:
    import adversec
except ModuleNotFoundError:
    sys.path.insert(0, str(Path.cwd().parent)); import adversec
import numpy as np, pandas as pd, io, contextlib
from adversec import config
import torch, joblib
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_class_weight
from adversec.models import CNN1D, train_cnn, build_random_forest
from adversec.experiments import attack as atk
from adversec.experiments.defense import adversarial_train_cnn
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'; print('device:', DEVICE)
DATASETS = ['ciciov2024', 'road']
def mf1(y, p, labels=None): return f1_score(y, p, labels=labels, average='macro', zero_division=0)

# load arrays once; robust-support = classes with >=2 test frames (computed, equal for both datasets)
data = {}
for name in DATASETS:
    a = np.load(config.PROCESSED_DIR / f'{name}_stage2_arrays.npz')
    classes = list(joblib.load(config.PROCESSED_DIR / f'{name}_label_encoder.joblib').classes_)
    cfg = config.load_dataset_config(name)
    counts = pd.Series(a['y_test']).value_counts()
    rs = [i for i in range(len(classes)) if counts.get(i, 0) >= 2]
    data[name] = dict(Xtr=a['X_train'], ytr=a['y_train'], Xte=a['X_test'].astype(np.float32),
                      yte=a['y_test'], classes=classes, cfg=cfg, rs=rs)
    print(f'{name}: robust-support classes = {[classes[i] for i in rs]}')

device: cuda
ciciov2024: robust-support classes = ['DoS', 'benign', 'spoofing-RPM']
road: robust-support classes = ['benign', 'fuzzing', 'max-speedometer', 'reverse-light-off', 'reverse-light-on']


## One run in detail (see the training and a single 2x2)

In [2]:
# ONE illustrative run
def one_run(name, seed=42, quiet_train=False):
    d = data[name]; Xtr, ytr, Xte, yte, classes, cfg, rs = (d['Xtr'], d['ytr'], d['Xte'], d['yte'], d['classes'], d['cfg'], d['rs'])
    cw = None
    if cfg.get('cnn_class_weights'):
        w = compute_class_weight('balanced', classes=np.unique(ytr), y=ytr)
        cw = torch.tensor(w, dtype=torch.float32, device=DEVICE)
    ctx = contextlib.redirect_stdout(io.StringIO()) if quiet_train else contextlib.nullcontext()
    with ctx:
        base = train_cnn(CNN1D(Xtr.shape[1], len(classes)), Xtr, ytr, n_epochs=50, device=DEVICE, class_weights=cw, random_seed=seed)
        defended = adversarial_train_cnn(Xtr, ytr, strategy='pgd', n_epochs=50, device=DEVICE, class_weights=cw, random_seed=seed)
    rf = build_random_forest(random_seed=seed).fit(Xtr, ytr)
    bcf = atk.wrap_cnn_for_art(base, Xtr.shape[1], len(classes), DEVICE)
    dcf = atk.wrap_cnn_for_art(defended, Xtr.shape[1], len(classes), DEVICE)
    Xt = atk.generate_pgd(bcf, Xte, 0.10)   # transfer (from baseline)
    Xw = atk.generate_pgd(dcf, Xte, 0.10)   # white-box (from defended)
    def f(clf, X): return mf1(yte, clf.predict(X).argmax(1), rs)
    return dict(base=(f(bcf, Xte), f(bcf, Xt), f(bcf, Xt)),
                defended=(f(dcf, Xte), f(dcf, Xt), f(dcf, Xw)),
                rf=(mf1(yte, rf.predict(Xte), rs), mf1(yte, rf.predict(Xt), rs), None))

for name in DATASETS:
    print('########## ' + name + ' ##########')
    r = one_run(name, seed=42, quiet_train=False)
    print(name + ': robust-support macro-F1, PGD eps=0.10')
    print(f"  {'model':15s}{'clean':>10}{'transfer':>10}{'white-box':>10}")
    for lab, key in [('baseline CNN','base'), ('defended CNN','defended'), ('Random Forest','rf')]:
        c, t, w = r[key]
        ws = 'n/a' if w is None else f'{w:.3f}'
        print(f"  {lab:15s}{c:>10.3f}{t:>10.3f}{ws:>10}")

########## ciciov2024 ##########
    epoch   1/50     loss 1.6402
    epoch   5/50     loss 0.1904
    epoch  10/50     loss 0.0311
    epoch  15/50     loss 0.0073
    epoch  20/50     loss 0.0037
    epoch  25/50     loss 0.0022
    epoch  30/50     loss 0.0016
    epoch  35/50     loss 0.0012
    epoch  40/50     loss 0.0012
    epoch  45/50     loss 0.0009
    epoch  50/50     loss 0.0009
    epoch   1/50     loss 1.6026
    epoch   5/50     loss 0.1219
    epoch  10/50     loss 0.0065
    epoch  15/50     loss 0.0023
    epoch  20/50     loss 0.0018
    epoch  25/50     loss 0.0014
    epoch  30/50     loss 0.0012
    epoch  35/50     loss 0.0010
    epoch  40/50     loss 0.0011
    epoch  45/50     loss 0.0010
    epoch  50/50     loss 0.0010


PGD - Batches:   0%|          | 0/120 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/120 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/120 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/120 [00:00<?, ?it/s]

...

PGD - Batches:   0%|          | 0/997 [00:00<?, ?it/s]

    augmented trainset: 31886 clean -> 191316 total (pgd strategy)
    epoch   1/50     loss 0.1528
    epoch   5/50     loss 0.0172
    epoch  10/50     loss 0.0090
    epoch  15/50     loss 0.0065
    epoch  20/50     loss 0.0053
    epoch  25/50     loss 0.0050
    epoch  30/50     loss 0.0043
    epoch  35/50     loss 0.0041
    epoch  40/50     loss 0.0037
    epoch  45/50     loss 0.0034
    epoch  50/50     loss 0.0035


PGD - Batches:   0%|          | 0/250 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/250 [00:00<?, ?it/s]

road: robust-support macro-F1, PGD eps=0.10
  model               clean  transfer white-box
  baseline CNN        0.997     0.331     0.331
  defended CNN        0.994     0.949     0.314
  Random Forest       1.000     0.139       n/a


## The decisive result - averaged over repeats (mean +/- std)

In [3]:
# AVERAGED decisive result: repeat the whole thing and report mean +/- std.
# CICIoV single runs are unstable (CUDA nondeterminism on tiny data), so only the average is trustworthy.
N_REPEATS = 10
keys = ['base_clean','base_atk','def_clean','def_transfer','def_whitebox','rf_clean','rf_atk']
agg = {name: {k: [] for k in keys} for name in DATASETS}
for r in range(N_REPEATS):
    for name in DATASETS:
        res = one_run(name, seed=42 + r, quiet_train=True)
        A = agg[name]
        A['base_clean'].append(res['base'][0]); A['base_atk'].append(res['base'][1])
        A['def_clean'].append(res['defended'][0]); A['def_transfer'].append(res['defended'][1]); A['def_whitebox'].append(res['defended'][2])
        A['rf_clean'].append(res['rf'][0]); A['rf_atk'].append(res['rf'][1])
    print(f'repeat {r + 1}/{N_REPEATS} done')

def ms(v): return f'{np.mean(v):.3f}+/-{np.std(v):.3f}'
print()
print('robust-support macro-F1, mean +/- std over %d repeats, PGD eps=0.10' % N_REPEATS)
print(f"  {'dataset':12s}{'model':15s}{'clean':>16}{'transfer':>16}{'white-box':>16}")
for name in DATASETS:
    A = agg[name]
    print(f"  {name:12s}{'baseline CNN':15s}{ms(A['base_clean']):>16}{ms(A['base_atk']):>16}{ms(A['base_atk']):>16}")
    print(f"  {'':12s}{'defended CNN':15s}{ms(A['def_clean']):>16}{ms(A['def_transfer']):>16}{ms(A['def_whitebox']):>16}")
    print(f"  {'':12s}{'Random Forest':15s}{ms(A['rf_clean']):>16}{ms(A['rf_atk']):>16}{'n/a':>16}")

PGD - Batches:   0%|          | 0/120 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/120 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/120 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/120 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/120 [00:00<?, ?it/s]

...

PGD - Batches:   0%|          | 0/997 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/997 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/250 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/250 [00:00<?, ?it/s]

repeat 10/10 done

robust-support macro-F1, mean +/- std over 10 repeats, PGD eps=0.10
  dataset     model                     clean        transfer       white-box
  ciciov2024  baseline CNN      0.793+/-0.035   0.287+/-0.028   0.287+/-0.028
              defended CNN      0.825+/-0.072   0.728+/-0.144   0.267+/-0.026
              Random Forest     0.868+/-0.022   0.321+/-0.065             n/a
  road        baseline CNN      0.997+/-0.001   0.330+/-0.026   0.330+/-0.026
              defended CNN      0.996+/-0.002   0.829+/-0.105   0.233+/-0.088
              Random Forest     1.000+/-0.000   0.163+/-0.060             n/a


## Save the decisive defence result (for report writing)
Writes `results/<name>_defence_results.json`: robust-support macro-F1 (mean ± std over `N_REPEATS`) for baseline CNN, defended CNN (transfer + white-box) and Random Forest, under PGD eps=0.10.

In [4]:
import json

def ms(v):
    arr = np.array(v, dtype=float)
    return {'mean': float(arr.mean()), 'std': float(arr.std()), 'runs': [float(x) for x in arr]}

config.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
for name in DATASETS:
    A, d = agg[name], data[name]
    report = {
        'dataset': name,
        'method': 'two_threat_model_averaged',
        'description': (
            'Robust-support macro-F1 under PGD eps=0.10, averaged over N_REPEATS seeded runs. '
            'transfer = attack crafted from the baseline CNN, fed to every model. '
            'white_box = attack crafted against the model it hits (CNN-defended only; '
            'the Random Forest has no gradients so white-box does not apply to it).'
        ),
        'attack_eps': 0.10,
        'n_repeats': N_REPEATS,
        'robust_support_classes': [d['classes'][i] for i in d['rs']],
        'results': {
            'baseline_cnn': {'clean': ms(A['base_clean']), 'attack': ms(A['base_atk'])},
            'defended_cnn': {'clean': ms(A['def_clean']), 'transfer': ms(A['def_transfer']), 'white_box': ms(A['def_whitebox'])},
            'random_forest': {'clean': ms(A['rf_clean']), 'attack': ms(A['rf_atk'])},
        },
    }
    path = config.RESULTS_DIR / f'{name}_defence_results.json'
    path.write_text(json.dumps(report, indent=2))
    print('saved ->', path)

saved -> /home/koala/lab/adversec/results/ciciov2024_defence_results.json
saved -> /home/koala/lab/adversec/results/road_defence_results.json


## Is the defence effect actually significant? (paired t-test across repeats)

Comparing means alone (e.g. "0.192 vs 0.327") doesn't say whether that gap is real or just
noise across repeats. Each repeat trains baseline and defended models from the *same* seed,
so `defended[r] - baseline[r]` is a proper paired comparison — a paired t-test on those
per-repeat differences is the right test here, and is much more powerful than comparing two
independent means with overlapping standard deviations.

In [5]:
import json
from scipy import stats as sstats

sig_results = {}
for name in DATASETS:
    d = json.loads((config.RESULTS_DIR / f'{name}_defence_results.json').read_text())
    r = d['results']
    base_atk = r['baseline_cnn']['attack']['runs']
    def_transfer = r['defended_cnn']['transfer']['runs']
    def_whitebox = r['defended_cnn']['white_box']['runs']

    t_transfer, p_transfer = sstats.ttest_rel(def_transfer, base_atk)
    t_whitebox, p_whitebox = sstats.ttest_rel(def_whitebox, base_atk)
    sig_results[name] = {
        'n_repeats': len(base_atk),
        'transfer_vs_undefended': {
            'mean_diff': float(np.mean(def_transfer) - np.mean(base_atk)),
            't_statistic': float(t_transfer),
            'p_value': float(p_transfer),
            'significant_at_0.05': bool(p_transfer < 0.05),
        },
        'white_box_vs_undefended': {
            'mean_diff': float(np.mean(def_whitebox) - np.mean(base_atk)),
            't_statistic': float(t_whitebox),
            'p_value': float(p_whitebox),
            'significant_at_0.05': bool(p_whitebox < 0.05),
        },
    }

print("paired t-test (defended vs undefended, same seed per repeat) -- computed from the saved runs, no retraining needed")
print(f"  {'dataset':12s}{'comparison':22s}{'n':>4}{'mean diff':>12}{'t':>8}{'p':>10}   significant?")
for name in DATASETS:
    s = sig_results[name]
    for label, key in [('transfer', 'transfer_vs_undefended'), ('white-box', 'white_box_vs_undefended')]:
        v = s[key]
        print(f"  {name:12s}{label:22s}{s['n_repeats']:>4}{v['mean_diff']:>+12.3f}{v['t_statistic']:>8.2f}{v['p_value']:>10.4f}   "
              f"{'YES' if v['significant_at_0.05'] else 'no'}")

paired t-test (defended vs undefended, same seed per repeat) -- computed from the saved runs, no retraining needed
  dataset     comparison               n   mean diff       t         p   significant?
  ciciov2024  transfer                10      +0.442    9.58    0.0000   YES
  ciciov2024  white-box               10      -0.020   -1.22    0.2549   no
  road        transfer                10      +0.499   13.21    0.0000   YES
  road        white-box               10      -0.098   -3.02    0.0145   YES


## Save the significance results

Merges into the `<name>_defence_results.json` already written above, adding a
`paired_significance_vs_undefended` section — doesn't touch anything already there.

In [6]:
for name in DATASETS:
    path = config.RESULTS_DIR / f'{name}_defence_results.json'
    existing = json.loads(path.read_text())
    existing['paired_significance_vs_undefended'] = sig_results[name]
    path.write_text(json.dumps(existing, indent=2))
    print('updated ->', path)

updated -> /home/koala/lab/adversec/results/ciciov2024_defence_results.json
updated -> /home/koala/lab/adversec/results/road_defence_results.json


In [7]:
# Standalone setup for the Madry section: everything the cells below need, so this
# section can be run on its own (after cell 1 only) without first re-running the
# static AT section above -- that section's results are already saved to disk.
import json
from scipy import stats as sstats
N_REPEATS = 10
def ms(v): return f'{np.mean(v):.3f}+/-{np.std(v):.3f}'

## Does the *iterative* (Madry-style) form of AT survive white-box, where the static form failed?

The defended CNN above (`defended_cnn`) is trained by augmenting the clean set with adversarial
examples crafted **once** from the frozen, undefended baseline (`adversarial_train_cnn`,
strategy='pgd') -- a static recipe. The literature draws a sharp line between this kind of
adversarial training (Kurakin et al.: single-shot / non-adaptive AT can give a false sense of
robustness) and the min-max form (Madry et al. 2018: adversarial examples crafted every batch
against the model's *current* weights), which is the form specifically claimed to survive a
white-box attacker.

This section repeats the exact same two-threat-model test (`transfer` + `white-box`, PGD
eps=0.10, robust-support macro-F1, averaged over the same `N_REPEATS` seeds) with
`madry_adversarial_train_cnn` in place of `adversarial_train_cnn`, so the two AT recipes are
compared under identical conditions. Train-time PGD uses fewer steps than eval-time PGD
(`config.MADRY_TRAIN_MAX_ITER=7` vs `config.PGD_MAX_ITER=40`) -- standard practice (Madry et
al. use 7 steps at CIFAR-10 train time, more steps at test time) to keep the inner loop
tractable without weakening the attack used to *measure* robustness afterwards. Unlike the
static recipe, training here is on adversarial batches only (no clean augmentation), matching
the canonical Madry et al. formulation -- the property under test is specifically whether the
min-max form of AT survives white-box, not the augmented-training recipe.

In [8]:
from adversec.experiments.defense import madry_adversarial_train_cnn

def one_run_madry(name, seed=42, quiet_train=False):
    d = data[name]; Xtr, ytr, Xte, yte, classes, cfg, rs = (d['Xtr'], d['ytr'], d['Xte'], d['yte'], d['classes'], d['cfg'], d['rs'])
    cw = None
    if cfg.get('cnn_class_weights'):
        w = compute_class_weight('balanced', classes=np.unique(ytr), y=ytr)
        cw = torch.tensor(w, dtype=torch.float32, device=DEVICE)
    ctx = contextlib.redirect_stdout(io.StringIO()) if quiet_train else contextlib.nullcontext()
    with ctx:
        base = train_cnn(CNN1D(Xtr.shape[1], len(classes)), Xtr, ytr, n_epochs=50, device=DEVICE, class_weights=cw, random_seed=seed)
        defended = madry_adversarial_train_cnn(Xtr, ytr, n_epochs=50, device=DEVICE, class_weights=cw, random_seed=seed)
    bcf = atk.wrap_cnn_for_art(base, Xtr.shape[1], len(classes), DEVICE)
    dcf = atk.wrap_cnn_for_art(defended, Xtr.shape[1], len(classes), DEVICE)
    Xt = atk.generate_pgd(bcf, Xte, 0.10)   # transfer (from baseline)
    Xw = atk.generate_pgd(dcf, Xte, 0.10)   # white-box (from madry-defended)
    def f(clf, X): return mf1(yte, clf.predict(X).argmax(1), rs)
    return dict(base=(f(bcf, Xte), f(bcf, Xt), f(bcf, Xt)),
                defended=(f(dcf, Xte), f(dcf, Xt), f(dcf, Xw)))

for name in DATASETS:
    print('########## ' + name + ' (madry) ##########')
    r = one_run_madry(name, seed=42, quiet_train=False)
    print(name + ': robust-support macro-F1, PGD eps=0.10, madry-defended')
    print(f"  {'model':19s}{'clean':>10}{'transfer':>10}{'white-box':>10}")
    for lab, key in [('baseline CNN', 'base'), ('madry-defended CNN', 'defended')]:
        c, t, w = r[key]
        print(f"  {lab:19s}{c:>10.3f}{t:>10.3f}{w:>10.3f}")

########## ciciov2024 (madry) ##########
    epoch   1/50     loss 1.6022
    epoch   5/50     loss 0.1946
    epoch  10/50     loss 0.0275
    epoch  15/50     loss 0.0069
    epoch  20/50     loss 0.0037
    epoch  25/50     loss 0.0023
    epoch  30/50     loss 0.0018
    epoch  35/50     loss 0.0014
    epoch  40/50     loss 0.0014
    epoch  45/50     loss 0.0011
    epoch  50/50     loss 0.0011
    [madry] epoch   1/50     loss 1.6572
    [madry] epoch   5/50     loss 0.4570
    [madry] epoch  10/50     loss 0.3242
    [madry] epoch  15/50     loss 0.2410
    [madry] epoch  20/50     loss 0.1694
    [madry] epoch  25/50     loss 0.1154
    [madry] epoch  30/50     loss 0.0873
    [madry] epoch  35/50     loss 0.0760
    [madry] epoch  40/50     loss 0.0319
    [madry] epoch  45/50     loss 0.0176
    [madry] epoch  50/50     loss 0.0115


PGD - Batches:   0%|          | 0/23 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/23 [00:00<?, ?it/s]

ciciov2024: robust-support macro-F1, PGD eps=0.10, madry-defended
  model                   clean  transfer white-box
  baseline CNN            0.766     0.315     0.315
  madry-defended CNN      0.796     0.370     0.287
########## road (madry) ##########
    epoch   1/50     loss 0.3182
    epoch   5/50     loss 0.0414
    epoch  10/50     loss 0.0185
    epoch  15/50     loss 0.0145
    epoch  20/50     loss 0.0123
    epoch  25/50     loss 0.0103
    epoch  30/50     loss 0.0087
    epoch  35/50     loss 0.0077
    epoch  40/50     loss 0.0063
    epoch  45/50     loss 0.0082
    epoch  50/50     loss 0.0057
    [madry] epoch   1/50     loss 0.4444
    [madry] epoch   5/50     loss 0.1083
    [madry] epoch  10/50     loss 0.1072
    [madry] epoch  15/50     loss 0.1068
    [madry] epoch  20/50     loss 0.1020
    [madry] epoch  25/50     loss 0.1015
    [madry] epoch  30/50     loss 0.1003
    [madry] epoch  35/50     loss 0.0997
    [madry] epoch  40/50     loss 0.1026
    [madry]

PGD - Batches:   0%|          | 0/250 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/250 [00:00<?, ?it/s]

road: robust-support macro-F1, PGD eps=0.10, madry-defended
  model                   clean  transfer white-box
  baseline CNN            0.998     0.297     0.297
  madry-defended CNN      0.767     0.472     0.549


## The decisive Madry result -- averaged over the same repeats (mean +/- std)

In [9]:
# AVERAGED madry result: identical structure/seeds to the static AT averaging above.
keys_madry = ['base_clean', 'base_atk', 'def_clean', 'def_transfer', 'def_whitebox']
agg_madry = {name: {k: [] for k in keys_madry} for name in DATASETS}
for r in range(N_REPEATS):
    for name in DATASETS:
        res = one_run_madry(name, seed=42 + r, quiet_train=True)
        A = agg_madry[name]
        A['base_clean'].append(res['base'][0]); A['base_atk'].append(res['base'][1])
        A['def_clean'].append(res['defended'][0]); A['def_transfer'].append(res['defended'][1]); A['def_whitebox'].append(res['defended'][2])
    print(f'repeat {r + 1}/{N_REPEATS} done (madry)')

print() 
print('robust-support macro-F1, mean +/- std over %d repeats, PGD eps=0.10, MADRY-defended' % N_REPEATS)
print(f"  {'dataset':12s}{'model':19s}{'clean':>16}{'transfer':>16}{'white-box':>16}")
for name in DATASETS:
    A = agg_madry[name]
    print(f"  {name:12s}{'baseline CNN':19s}{ms(A['base_clean']):>16}{ms(A['base_atk']):>16}{ms(A['base_atk']):>16}")
    print(f"  {'':12s}{'madry-defended CNN':19s}{ms(A['def_clean']):>16}{ms(A['def_transfer']):>16}{ms(A['def_whitebox']):>16}")

PGD - Batches:   0%|          | 0/23 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/23 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/250 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/250 [00:00<?, ?it/s]

repeat 1/10 done (madry)


...

PGD - Batches:   0%|          | 0/23 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/23 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/250 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/250 [00:00<?, ?it/s]

repeat 10/10 done (madry)

robust-support macro-F1, mean +/- std over 10 repeats, PGD eps=0.10, MADRY-defended
  dataset     model                         clean        transfer       white-box
  ciciov2024  baseline CNN          0.785+/-0.030   0.286+/-0.020   0.286+/-0.020
              madry-defended CNN    0.791+/-0.040   0.471+/-0.109   0.389+/-0.078
  road        baseline CNN          0.997+/-0.002   0.286+/-0.103   0.286+/-0.103
              madry-defended CNN    0.766+/-0.001   0.513+/-0.044   0.551+/-0.066


## Save the Madry result (merges into the same `<name>_defence_results.json`)

Adds a `defended_cnn_madry` section alongside the existing `defended_cnn` (static AT) section
-- doesn't touch anything already there.

In [10]:
def ms_full(v):
    arr = np.array(v, dtype=float)
    return {'mean': float(arr.mean()), 'std': float(arr.std()), 'runs': [float(x) for x in arr]}

for name in DATASETS:
    A = agg_madry[name]
    path = config.RESULTS_DIR / f'{name}_defence_results.json'
    existing = json.loads(path.read_text())
    existing['defended_cnn_madry'] = {
        'description': (
            'Same robust-support macro-F1 protocol as defended_cnn, but the defended CNN is '
            'trained with madry_adversarial_train_cnn: PGD adversarial examples crafted every '
            f"batch against the model's CURRENT weights (train-time PGD max_iter="
            f"{config.MADRY_TRAIN_MAX_ITER}, eps={config.MADRY_TRAIN_EPSILON}), not a fixed set "
            "crafted once from a frozen baseline. Pure adversarial batches (no clean "
            "augmentation), matching the canonical Madry et al. formulation."
        ),
        'train_time_pgd_max_iter': config.MADRY_TRAIN_MAX_ITER,
        'train_time_epsilon': config.MADRY_TRAIN_EPSILON,
        'eval_attack_eps': 0.10,
        'n_repeats': N_REPEATS,
        # The baseline trained alongside the Madry model in the SAME run, seed for seed.
        # The significance test below pairs against this, not against the static section's
        # baseline -- same seeds, but a different trained model under CUDA nondeterminism,
        # so pairing across sections is not a true paired comparison.
        'baseline_clean': ms_full(A['base_clean']),
        'baseline_attack': ms_full(A['base_atk']),
        'clean': ms_full(A['def_clean']),
        'transfer': ms_full(A['def_transfer']),
        'white_box': ms_full(A['def_whitebox']),
    }
    path.write_text(json.dumps(existing, indent=2))
    print('updated ->', path)

updated -> /home/koala/lab/adversec/results/ciciov2024_defence_results.json
updated -> /home/koala/lab/adversec/results/road_defence_results.json


## Is the iterative (Madry) form significantly different from the static form, white-box?

Two comparisons, both paired by repeat (same seed `42 + r` trains both the static-run's and the
madry-run's baseline/defended pair for repeat `r`, matching the paired design already used
above):

- **madry white-box vs the undefended baseline under attack** -- does the min-max form help at
  all, white-box?
- **madry white-box vs the static defended white-box** -- the comparison that actually answers
  the original question: is the iterative form significantly different from the static form
  that was shown to collapse white-box?

In [11]:
sig_madry = {}
for name in DATASETS:
    path = config.RESULTS_DIR / f'{name}_defence_results.json'
    d = json.loads(path.read_text())
    # Pair within-run: the baseline from the Madry section, not the static section's.
    base_atk = d['defended_cnn_madry']['baseline_attack']['runs']
    # NOTE: static_wb comes from the static section, madry_wb from the Madry section --
    # two separate training passes. Same seeds, but not the same run, so this particular
    # comparison is only approximately paired. Notebook 07 trains baseline, static AT and
    # Madry AT together in ONE call per seed, so its 'whitebox_madry_vs_static_pgd' is the
    # properly within-run paired version of this test -- prefer that number when quoting
    # 'iterative AT beats static AT under white-box'.
    static_wb = d['results']['defended_cnn']['white_box']['runs']
    madry_wb = d['defended_cnn_madry']['white_box']['runs']

    t_vs_base, p_vs_base = sstats.ttest_rel(madry_wb, base_atk)
    t_vs_static, p_vs_static = sstats.ttest_rel(madry_wb, static_wb)
    sig_madry[name] = {
        'n_repeats': len(base_atk),
        'madry_white_box_vs_undefended': {
            'mean_diff': float(np.mean(madry_wb) - np.mean(base_atk)),
            't_statistic': float(t_vs_base), 'p_value': float(p_vs_base),
            'significant_at_0.05': bool(p_vs_base < 0.05),
        },
        'madry_white_box_vs_static_white_box': {
            'mean_diff': float(np.mean(madry_wb) - np.mean(static_wb)),
            't_statistic': float(t_vs_static), 'p_value': float(p_vs_static),
            'significant_at_0.05': bool(p_vs_static < 0.05),
        },
    }

print("paired t-test -- madry-defended white-box vs undefended-under-attack (paired within-run),"
      " and vs static-defended white-box")
print(f"  {'dataset':12s}{'comparison':32s}{'n':>4}{'mean diff':>12}{'t':>8}{'p':>10}   significant?")
for name in DATASETS:
    s = sig_madry[name]
    for label, key in [('vs undefended (attacked)', 'madry_white_box_vs_undefended'),
                        ('vs static AT (white-box)', 'madry_white_box_vs_static_white_box')]:
        v = s[key]
        print(f"  {name:12s}{label:32s}{s['n_repeats']:>4}{v['mean_diff']:>+12.3f}{v['t_statistic']:>8.2f}{v['p_value']:>10.4f}   "
              f"{'YES' if v['significant_at_0.05'] else 'no'}")

paired t-test -- madry-defended white-box vs undefended-under-attack (paired within-run), and vs static-defended white-box
  dataset     comparison                         n   mean diff       t         p   significant?
  ciciov2024  vs undefended (attacked)          10      +0.104    4.01    0.0031   YES
  ciciov2024  vs static AT (white-box)          10      +0.122    5.09    0.0007   YES
  road        vs undefended (attacked)          10      +0.265    6.97    0.0001   YES
  road        vs static AT (white-box)          10      +0.318    9.46    0.0000   YES


## Save the Madry significance results

In [12]:
for name in DATASETS:
    path = config.RESULTS_DIR / f'{name}_defence_results.json'
    existing = json.loads(path.read_text())
    existing['paired_significance_madry'] = sig_madry[name]
    path.write_text(json.dumps(existing, indent=2))
    print('updated ->', path)

updated -> /home/koala/lab/adversec/results/ciciov2024_defence_results.json
updated -> /home/koala/lab/adversec/results/road_defence_results.json
